# MusicBrainz Exploration — EDM/Music Sources
No account/API key required for read-only search. Just need a User-Agent header identifying the app.

In [1]:
import requests
import pandas as pd
import time
import logging
import os
import ast
from collections import Counter

from data_ravers_utils.file_handler import save_df_pickle, read_df_pickle, save_df_as_csv
from data_ravers_utils import eda_utils

In [2]:
# API Configuration Constants
HEADERS = {"User-Agent": "AGIes/0.1 (irum.shehryar@gmail.com)"}
BASE_URL = "https://musicbrainz.org/ws/2/release-group/"
DEFAULT_TAG = "edm"
FETCH_LIMIT = 100
RATE_LIMIT_DELAY = 1.2
REQUEST_TIMEOUT=30 

In [3]:
def get_mb_tag_total_count(tag_name=DEFAULT_TAG):

    """
    Fetch the live total record count for a specific tag on MusicBrainz.
    """
    params={"query": f"tag:{tag_name}", "fmt": "json", "limit": 1}

    try:
        response = requests.get(BASE_URL, params=params, headers=HEADERS, timeout=10)
        response.raise_for_status()
        total_count=response.json().get("count",0)
        print(f"Total records for tag '{tag_name}': {total_count}")
        return total_count
    except requests.exceptions.RequestException as e:
        logging.error(f"Failed to query MusicBrainz count for tag '{tag_name}': {e}")
        raise 

In [4]:
total_edm_records = get_mb_tag_total_count()


Total records for tag 'edm': 5746


## Fetch a sample batch
Pulling a small sample (not the full 5,746 records) for initial exploration, respecting the 1 request/second rate limit.

In [5]:
def fetch_raw_musicbrainz_sample(sample_limit, tag=DEFAULT_TAG, limit=FETCH_LIMIT):
    """Fetch a small sample batch for initial EDA without full collection wait times."""
    raw_sample = []
    
    for offset in range(0, sample_limit, limit):
        params = {
            "query": f"tag:{tag}",
            "fmt": "json",
            "limit": FETCH_LIMIT,
            "offset": offset
        }
        
        try:
            response = requests.get(BASE_URL, params=params, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            data = response.json()
            raw_sample.extend(data.get("release-groups", []))
            print(f"Fetched sample offset {offset} ({len(raw_sample)} records)")
        
        except requests.exceptions.Timeout:
            logging.warning(f"Request timed out at offset {offset}. Skipping batch.")
        except requests.exceptions.RequestException as e:
            logging.warning(f"Error fetching offset {offset}: {e}")
        except ValueError as e:
            logging.warning(f"Failed to parse JSON response at offset {offset}: {e}")
            
        time.sleep(RATE_LIMIT_DELAY)
        
    return raw_sample



In [6]:
# Fetch raw JSON list from API
raw_sample_data = fetch_raw_musicbrainz_sample(sample_limit=500)
df_sample=pd.DataFrame({"raw_payload": raw_sample_data})
print(f"Collected {len(df_sample)} records for exploration.")

Fetched sample offset 0 (100 records)


Collected 100 records for exploration.


In [7]:
# 1. Ensure the 'data' directory exists locally
os.makedirs("data", exist_ok=True)
# Save raw sample binary pickle
save_df_pickle(df=df_sample, filename="musicbrainz_edm_raw_sample")
# Save flat CSV for quick manual viewing
save_df_as_csv(df=df_sample, filename="musicbrainz_edm_raw_sample")
print("Directory created and raw sample files successfully saved!")


Directory created and raw sample files successfully saved!


## Reload saved sample and run EDA

In [8]:
df_raw=read_df_pickle("musicbrainz_edm_raw_sample")
print("DataFrame Shape:", df_raw.shape)
df_raw

DataFrame Shape: (100, 1)


,raw_payload
0,"{'id': '3bf604a1-5039-4ead-ac3e-1959884c9c97',..."
1,"{'id': '4dc4fb78-e096-3f7b-9334-9defe9890ef8',..."
2,"{'id': '3b766d4f-7b6b-44a3-9b52-bff158484f40',..."
3,"{'id': '8a787daf-2a37-45a1-9ae1-53197eb79541',..."
4,"{'id': 'ddfc24ec-3376-4ad0-8279-ca47d579fcee',..."
...,...
95,"{'id': '01def791-9922-4752-b715-0b06b63f0576',..."
96,"{'id': '5057c53e-c387-42ed-a113-1c73d6afc0ba',..."
97,"{'id': 'd5296971-21e3-4483-b222-0cae32fc9408',..."
98,"{'id': '695215db-46e9-4999-aa93-a8813f3a0a82',..."


In [13]:
# Flatten raw JSON into a temporary DataFrame for EDA
df_samples = pd.json_normalize(df_raw["raw_payload"])
# 2. Convert list and dictionary columns to strings so unique/nunique operations don't fail
for col in df_samples.columns:
    if df_samples[col].apply(lambda x: isinstance(x, (list, dict))).any():
        df_samples[col] = df_samples[col].astype(str)


In [15]:
# Generate report on 500 samples
eda_utils.print_eda_report(df_samples)

================= Dataset =================
Dataset has shape (100, 14)

Dataset has numerical data in columns: ['score', 'count']
- Column "count" has 6 unique values.
  -- Unique values are:
 [1 3 2 4 5 9]
- Column "score" has 5 unique values.
  -- Unique values are:
 [100  96  95  90  87]

Dataset has categorical data in columns: ['id', 'type-id', 'primary-type-id', 'artist-credit-id', 'title', 'first-release-date', 'primary-type', 'secondary-types', 'secondary-type-ids', 'artist-credit', 'releases', 'tags']
- Column "id" has 100 unique values.
- Column "type-id" has 7 unique values.
  -- Unique values are:
 <StringArray>
['d6038452-8ee0-3f68-affc-2de9a1ede0b9',
 'f529b476-6e62-324f-b0aa-1f3e33d313fc',
 '6d0c5bf6-7a33-3420-a519-44fc63eedebf',
 'dd2a21e1-0c00-3729-a7a0-de60b84eb5d1',
 '0c60f497-ff81-3818-befd-abfc84a4858b',
 '6fd474e2-6b58-3102-9d17-d6f7eb7da0a0',
 '22a628ad-c082-3c4f-b1b6-d41665107b88']
Length: 7, dtype: str
- Column "primary-type-id" has 3 unique values.
  -- Uniqu

## Extract and count individual tags
The `tags` column is stored as a stringified list — parse it back into real tag names and count frequency across the sample.

In [16]:
def extract_tag_names(raw_tags_str):
    """Convert the stringified tags list back into just a list of tag names."""
    try:
        tags_list = ast.literal_eval(raw_tags_str)
        return [t['name'] for t in tags_list]
    except (ValueError, SyntaxError, TypeError):
        return []

In [17]:
df_samples['tag_names'] = df_samples['tags'].apply(extract_tag_names)

In [19]:
all_tags = [tag for tags_list in df_samples['tag_names'] for tag in tags_list]
tag_counts = Counter(all_tags)
pd.Series(tag_counts).sort_values(ascending=False).head(20)

edm               100
electronic         13
dance               4
pop                 4
trap edm            2
gaming edm          2
darkcore edm        1
emo                 1
english             1
trap                1
dance & edm         1
latvia              1
latvian edm         1
lithuania           1
lithuanian          1
hardstyle           1
lithuanian edm      1
jumpstyle           1
alternative         1
house               1
dtype: int64

In [ ]:
## Full-collection function (not run)
Kept for future use if the team decides to pull the complete dataset (5,746 records) rather than a sample. Includes retry/backoff handling for 503 errors.

In [ ]:
"""
Paginate and store complete unparsed records with backoff retry handling for 503 errors."""

"""
def fetch_raw_musicbrainz_records(total_count, tag=DEFAULT_TAG):
    
    raw_records = []
    for offset in range(0, total_count, FETCH_LIMIT):
        params = {"query": f"tag:{tag}", "fmt": "json", "limit": FETCH_LIMIT, "offset": offset}
        success = False
        retries = 0
        
        while not success and retries < 3:
            try:
                res = requests.get(BASE_URL, params=params, headers=HEADERS, timeout=REQUEST_TIMEOUT)
                res.raise_for_status()
                raw_records.extend(res.json().get("release-groups", []))
                success = True
            except requests.exceptions.HTTPError as e:
                if res.status_code == 503:
                    retries += 1
                    print(f"HTTP 503 hit at offset {offset}. Backing off for {retries * 5} seconds...")
                    time.sleep(retries * 5)
                else:
                    print(f"HTTP Error at offset {offset}: {e}")
                    break
            except Exception as e:
                print(f"Request failed at offset {offset}: {e}")
                break
                
        time.sleep(RATE_LIMIT_DELAY)
    return raw_records

# Execute full bulk download when ready
# raw_full_data = fetch_raw_musicbrainz_records(total_count=total_edm_records)"""
